# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A1704/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of analysis and time window

**One row means:** one daily performance record for a specific content item belonging to a specific pseudonymized client on a particular reporting date.

**Table used:** `fact_content_daily_performance`, using the March 2026 partition (`month=2026-03`).

**Time window:** 1–31 March 2026, subject to the dates actually present in the partition. March is the development slice, not the sealed June 2026 sample.

**Prediction goal:** rank content items for editorial review based on available performance signals. A future decline label must be defined using an outcome window that occurs after the scoring moment; the current schema alone does not establish such a label.

**Deliberate exclusion:** exclude `report_date`, identifiers, and availability flags from the predictive feature set. Use identifiers and flags for grouping, joining, or filtering—not as predictive signals. Do not use future outcomes or label-derived fields as features.

In [63]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the expected grain and March date coverage.
assert q1.empty, "Duplicate report_date × client × content combinations found."

print("Unit of analysis:")
print("One row = one client-content pair on one reporting date.")
print("\nMarch reporting window:")
display(q2)
print("\nGrain check passed: no duplicate combinations found.")


Unit of analysis:
One row = one client-content pair on one reporting date.

March reporting window:


,row_count,first_report_date,last_report_date,distinct_report_dates
0,9841378,2026-03-01,2026-03-31,31



Grain check passed: no duplicate combinations found.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `sessions_organic`.

**Label:** `is_declining_label`, defined using the change in average daily GSC clicks from March to April 2026. This is a proxy for declining search performance, not a definitive measure of content quality.

**Context fields:** `report_date`, `client_hash_id`, `content_hash_id`, `gsc_data_available`, and `ga4_data_available`. These support time-window checks, grouping, joins, and availability filtering.

**Deliberate exclusion:** I will not use future April outcomes or `is_declining_label` as predictive features. Identifiers and availability flags are used for data handling, not as predictive signals. The label is constructed only after the March feature window is defined.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Findings and limitations

**Data contract**
- One source row represents a content item for a pseudonymized client on a reporting date.
- March 2026 is the feature-development window. April 2026 provides the future outcome window.
- After applying the reporting-coverage rules and matching client-content pairs across both months, the modelling dataset contains 3,560 pairs.
- `is_declining_label` equals 1 when April average daily GSC clicks are lower than March average daily GSC clicks; otherwise it equals 0. This is a proxy for declining search performance, not a definitive measure of content quality.

**Data limitations**
- Reporting coverage varies by client and content item, so the eligible modelling dataset is a selected subset of the warehouse.
- March features are aggregated over days when both GSC and GA4 are available, while April outcomes use days when GSC is available. These reporting-day sets can differ.
- Zero-filled values must not be interpreted as genuine zero activity when the corresponding source is unavailable.
- The target measures a change in average daily clicks; it does not establish why performance changed or whether content needs refreshing.
- The baseline model's balanced accuracy is approximately 0.503 and ROC-AUC is approximately 0.561 on a random holdout. This is weak evidence of predictive signal and is not sufficient for production use.
- The random train/test split does not establish generalization to unseen clients or future time periods.

**Leakage lesson**
A deliberately added `future_label_leak` feature directly copies the target and produces perfect scores. This is invalid because the target would not be known at prediction time. The leaked feature was removed, leaving the five approved features and the original honest baseline scores.

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [38]:

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is missing. Check the Secrets panel and notebook access."

print("Hugging Face secret found. Token kept private.")


Hugging Face secret found. Token kept private.


In [39]:

%pip -q install duckdb


In [40]:

import duckdb

con = duckdb.connect()

# Authenticate without printing or hardcoding the token.
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connected and Hugging Face authentication configured.")


DuckDB connected and Hugging Face authentication configured.


In [41]:

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is missing from Colab Secrets."

con = duckdb.connect()

# Install/load DuckDB's HTTP filesystem support.
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Use DuckDB's documented Hugging Face secret syntax.
con.execute(
    "CREATE OR REPLACE SECRET hf_token "
    "(TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB HTTP support and Hugging Face secret configured.")


DuckDB HTTP support and Hugging Face secret configured.


In [ ]:

%pip -q install -U huggingface_hub duckdb


In [43]:

%pip install -q "huggingface_hub>=0.34,<1.0"


In [44]:

import huggingface_hub

print("Installed version:", huggingface_hub.__version__)

try:
    from huggingface_hub import hf_hub_download
    print("Download function is available.")
except ImportError as e:
    print("Import error:", e)


Installed version: 0.36.2
Download function is available.


In [45]:

from google.colab import userdata
from huggingface_hub import hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN").strip()
assert HF_TOKEN, "HF_TOKEN is empty."

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print("March partition downloaded successfully.")
print("Local file:", file_path)


March partition downloaded successfully.
Local file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [47]:

import duckdb

con = duckdb.connect()

schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [file_path]
).fetchdf()

display(schema[["column_name", "column_type"]])


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [48]:

# Query 1 — Verify the proposed row grain.
# Expected grain: report_date × client_hash_id × content_hash_id

q1 = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS records_at_grain
    FROM read_parquet(?)
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""", [file_path]).fetchdf()

print("Query 1 — Duplicate grain combinations (zero rows means no duplicates found):")
display(q1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Duplicate grain combinations (zero rows means no duplicates found):


,report_date,client_hash_id,content_hash_id,records_at_grain


In [49]:

# Query 2 — Verify row count and date span for March.

q2 = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS distinct_report_dates
    FROM read_parquet(?)
""", [file_path]).fetchdf()

print("Query 2 — March row count and date span:")
display(q2)


Query 2 — March row count and date span:


,row_count,first_report_date,last_report_date,distinct_report_dates
0,9841378,2026-03-01,2026-03-31,31


In [50]:

# Query 3 — Count records with confirmed Google Search Console availability.
# IS TRUE explicitly excludes FALSE and NULL.

q3 = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS rows_with_gsc_available,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS rows_with_both_sources_available
    FROM read_parquet(?)
""", [file_path]).fetchdf()

print("Query 3 — Availability-filter survivor counts:")
display(q3)


Query 3 — Availability-filter survivor counts:


,total_rows,rows_with_gsc_available,rows_with_both_sources_available
0,9841378,3611061,364347


### Verification findings

1. **Grain:** The duplicate check returned zero rows. This supports the proposed grain of `report_date × client_hash_id × content_hash_id` for March 2026.
2. **Row count and date span:** The partition contains 9,841,378 rows, covering 31 distinct reporting dates from 2026-03-01 through 2026-03-31.
3. **Data availability:** Of the 9,841,378 rows, 3,611,061 have `gsc_data_available IS TRUE`, and 364,347 have both `gsc_data_available IS TRUE` and `ga4_data_available IS TRUE`.

**Limitation:** These findings describe the March 2026 partition only. Data availability is uneven, so models requiring both GSC and GA4 signals would have substantially fewer eligible rows than the full dataset.

In [51]:

# W03 — Candidate feature frame for March 2026
# This is a feature-availability audit, not a trained prediction model.

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic",
]

feature_frame = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        sessions_organic
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
""", [file_path]).fetchdf()

print("Feature-frame shape:", feature_frame.shape)
display(feature_frame.head())


Feature-frame shape: (3611061, 10)


,report_date,client_hash_id,content_hash_id,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,sessions_organic
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,<NA>,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,<NA>,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,<NA>,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,<NA>,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,<NA>,11,0,2.272727,<NA>,<NA>


### Candidate features and availability at decision time

- **`gsc_impressions`** — available at the scoring moment if Search Console data has been collected for the reporting date and `gsc_data_available IS TRUE`.
- **`gsc_clicks`** — available at the scoring moment under the same Search Console availability condition.
- **`gsc_avg_position`** — available when Search Console data is available; its interpretation may be limited for rows with no meaningful ranking observations.
- **`ga4_pageviews`** — usable only when `ga4_data_available IS TRUE`; otherwise, zero-filled or unavailable values must not be interpreted as genuine zero traffic.
- **`sessions_organic`** — usable only when `ga4_data_available IS TRUE`, subject to the dataset's collection and reporting lag.

**Feature-selection caveat:** This frame filters on GSC availability but retains rows where GA4 may be unavailable. Before model training, either restrict the training population to rows where both sources are available or use a documented missing-data strategy.

In [52]:

print("Rows in feature frame:", len(feature_frame))

print("\nAvailability flags:")
display(
    feature_frame[
        ["gsc_data_available", "ga4_data_available"]
    ].isna().sum().rename("missing_rows").to_frame()
)

print("\nRows with GA4 explicitly available:")
print(
    int(feature_frame["ga4_data_available"].eq(True).sum())
)

print("\nMissing values in candidate features:")
display(
    feature_frame[feature_columns]
    .isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)


Rows in feature frame: 3611061

Availability flags:


,missing_rows
gsc_data_available,0
ga4_data_available,1528366



Rows with GA4 explicitly available:
364347

Missing values in candidate features:


,missing_rows
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_pageviews,1528366
sessions_organic,1528366


In [53]:

# Restrict to rows with both data sources explicitly available.
model_frame = feature_frame.loc[
    feature_frame["gsc_data_available"].eq(True)
    & feature_frame["ga4_data_available"].eq(True)
].copy()

# The five candidate predictive features.
feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic",
]

print("Rows eligible for both sources:", len(model_frame))
print("Missing values in five features:")
display(model_frame[feature_columns].isna().sum().to_frame("missing_rows"))

display(model_frame[feature_columns].head())


Rows eligible for both sources: 364347
Missing values in five features:


,missing_rows
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_pageviews,0
sessions_organic,0


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,sessions_organic
8428,5,0,5.400000,1,1
8458,39,0,5.666667,2,0
8463,179,0,5.156425,2,0
8466,72,0,7.694444,1,0
8475,3282,1,6.167885,1,1


In [54]:

march_features = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_pageviews) AS ga4_pageviews,
        SUM(sessions_organic) AS sessions_organic,
        COUNT(DISTINCT report_date) AS observed_days
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""", [file_path]).fetchdf()

print("March client-content rows:", len(march_features))
print("Duplicate client-content pairs:",
      march_features.duplicated(
          ["client_hash_id", "content_hash_id"]
      ).sum())

display(march_features.head())


March client-content rows: 63856
Duplicate client-content pairs: 0


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,sessions_organic,observed_days
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,458.0,2.0,4.418032,16.0,10.0,9
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,3943.0,23.0,4.392897,59.0,39.0,25
2,client_65de48885f4ef01b,content_3c286ded8bd68120,2180.0,15.0,8.439390,31.0,21.0,19
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,503.0,8.0,5.531459,23.0,8.0,14
4,client_65de48885f4ef01b,content_ff867882e604fa96,24.0,0.0,2.850000,2.0,1.0,2


In [55]:

from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN is missing. Add it in Colab Secrets "
        "and enable notebook access."
    )

hf_token = hf_token.strip()
print("Hugging Face token loaded successfully.")


Hugging Face token loaded successfully.


In [56]:

from huggingface_hub import hf_hub_download
from pathlib import Path

april_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-04/data_0.parquet",
    token=hf_token,
)

print("April partition downloaded:", Path(april_path).name)
print("File exists:", Path(april_path).exists())


April partition downloaded: data_0.parquet
File exists: True


In [57]:

april_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS distinct_report_dates,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS rows_with_gsc_available,
        COUNT(DISTINCT report_date) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS dates_with_gsc_available
    FROM read_parquet(?)
""", [april_path]).fetchdf()

display(april_check)


,total_rows,first_report_date,last_report_date,distinct_report_dates,rows_with_gsc_available,dates_with_gsc_available
0,10424730,2026-04-01,2026-04-30,30,3901060,30


In [58]:

april_outcomes = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_clicks) AS april_avg_daily_clicks,
        COUNT(DISTINCT report_date) AS april_gsc_days
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""", [april_path]).fetchdf()

print("April client-content pairs:", len(april_outcomes))
print("Pairs with at least 20 GSC reporting days:",
      (april_outcomes["april_gsc_days"] >= 20).sum())

display(april_outcomes.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April client-content pairs: 194760
Pairs with at least 20 GSC reporting days: 115929


,client_hash_id,content_hash_id,april_avg_daily_clicks,april_gsc_days
0,client_62f4a7e64f5e0096,content_92bc8dfb830d0ade,0.0,15
1,client_62f4a7e64f5e0096,content_f1c085e5ea530266,0.0,25
2,client_62f4a7e64f5e0096,content_c4901b51a12b1c3b,0.0,24
3,client_62f4a7e64f5e0096,content_e68b30ce53db783f,0.0,30
4,client_62f4a7e64f5e0096,content_dd37ba6019bb33f7,0.0,19


In [59]:

import pandas as pd

# Keep pairs with sufficient reporting coverage in both months.
march_eligible = march_features.loc[
    march_features["observed_days"] >= 20
].copy()

april_eligible = april_outcomes.loc[
    april_outcomes["april_gsc_days"] >= 20
].copy()

# March average daily clicks, using days with both sources available.
march_eligible["march_avg_daily_clicks"] = (
    march_eligible["gsc_clicks"] / march_eligible["observed_days"]
)

# Join on the client-content pair.
model_data = march_eligible.merge(
    april_eligible,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

# Future outcome: 1 if average daily GSC clicks fell in April.
model_data["is_declining_label"] = (
    model_data["april_avg_daily_clicks"]
    < model_data["march_avg_daily_clicks"]
).astype(int)

print("March pairs with at least 20 observed days:", len(march_eligible))
print("April pairs with at least 20 GSC days:", len(april_eligible))
print("Matched eligible pairs:", len(model_data))
print("\nTarget distribution:")
display(
    model_data["is_declining_label"]
    .value_counts()
    .rename(index={0: "Not declining", 1: "Declining"})
    .to_frame("rows")
)

display(
    model_data[
        [
            "march_avg_daily_clicks",
            "april_avg_daily_clicks",
            "observed_days",
            "april_gsc_days",
            "is_declining_label",
        ]
    ].head()
)


March pairs with at least 20 observed days: 3563
April pairs with at least 20 GSC days: 115929
Matched eligible pairs: 3560

Target distribution:


,rows
is_declining_label,
Declining,2359
Not declining,1201


,march_avg_daily_clicks,april_avg_daily_clicks,observed_days,april_gsc_days,is_declining_label
0,0.920000,0.533333,25,30,1
1,7.275862,13.300000,29,30,0
2,2.785714,3.133333,28,30,0
3,4.758621,4.900000,29,30,0
4,5.214286,3.100000,28,30,1


In [60]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic",
]

X = model_data[feature_columns].copy()
y = model_data["is_declining_label"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

honest_model.fit(X_train, y_train)
honest_predictions = honest_model.predict(X_test)
honest_probabilities = honest_model.predict_proba(X_test)[:, 1]

honest_accuracy = accuracy_score(y_test, honest_predictions)
honest_balanced_accuracy = balanced_accuracy_score(y_test, honest_predictions)
honest_auc = roc_auc_score(y_test, honest_probabilities)

print(f"Honest accuracy:          {honest_accuracy:.3f}")
print(f"Honest balanced accuracy: {honest_balanced_accuracy:.3f}")
print(f"Honest ROC-AUC:           {honest_auc:.3f}")


Honest accuracy:          0.666
Honest balanced accuracy: 0.507
Honest ROC-AUC:           0.579


In [61]:

# INTENTIONAL LEAK: this feature directly copies the future-derived label.
X_leaky = X.copy()
X_leaky["future_label_leak"] = y

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leaky, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

leaky_model.fit(X_train_leak, y_train_leak)
leaky_predictions = leaky_model.predict(X_test_leak)
leaky_probabilities = leaky_model.predict_proba(X_test_leak)[:, 1]

print("INTENTIONALLY LEAKED MODEL — INVALID")
print(f"Accuracy:          {accuracy_score(y_test_leak, leaky_predictions):.3f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test_leak, leaky_predictions):.3f}")
print(f"ROC-AUC:           {roc_auc_score(y_test_leak, leaky_probabilities):.3f}")


INTENTIONALLY LEAKED MODEL — INVALID
Accuracy:          1.000
Balanced accuracy: 1.000
ROC-AUC:           1.000


In [62]:

# Restore the approved five-feature frame.
approved_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic",
]

X_final = model_data[approved_features].copy()

assert "future_label_leak" not in X_final.columns
assert len(X_final.columns) <= 5

print("Final feature count:", len(X_final.columns))
print("Final features:", X_final.columns.tolist())
print("Leaked feature retained:", "future_label_leak" in X_final.columns)
print(f"Honest accuracy retained:          {honest_accuracy:.3f}")
print(f"Honest balanced accuracy retained: {honest_balanced_accuracy:.3f}")
print(f"Honest ROC-AUC retained:           {honest_auc:.3f}")


Final feature count: 5
Final features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'sessions_organic']
Leaked feature retained: False
Honest accuracy retained:          0.666
Honest balanced accuracy retained: 0.507
Honest ROC-AUC retained:           0.579
